# 📖 Notebook 4: Partitioning Strategies

When a table grows to **hundreds of millions or billions of rows**, even indexed queries slow down. Partitioning splits a huge table into smaller, manageable pieces called **partitions**. PostgreSQL can then skip irrelevant partitions entirely — this is called **partition pruning**.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why partitioning matters for large tables
- How Range, List, and Hash partitioning work
- How partition pruning speeds up queries
- When to use each partitioning strategy

## The Pattern: BAD → BETTER → BEST

| Approach | Technique | Performance |
|----------|-----------|-------------|
| 🔴 BAD | Single huge table (millions of rows) | Slow scans, bloated indexes |
| 🟡 BETTER | Range partitioning (by date) | Fast time-based queries |
| 🟢 BEST | Right partition strategy for your access pattern | Optimal pruning |

## 🛠️ Setup

```bash
cd deep-dives/postgres
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import time
from tabulate import tabulate

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "postgres_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def run_sql(sql, params=None, fetch=True):
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql, params)
    result = cur.fetchall() if fetch and cur.description else None
    cols = [d[0] for d in cur.description] if cur.description else []
    conn.close()
    return result, cols

def explain(sql, params=None):
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {sql}", params)
    plan = cur.fetchall()
    conn.close()
    print("┌─── EXPLAIN ANALYZE ───────────────────────")
    for row in plan:
        print(f"│ {row[0]}")
    print("└───────────────────────────────────────────────────────")

def timed_query(sql, params=None, label="Query", runs=5):
    """Reuse one connection so timings measure the query, not the TCP handshake.
    A warm-up run is discarded so results reflect warm-cache performance."""
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    try:
        cur.execute(sql, params)
        cur.fetchall()
        times = []
        for _ in range(runs):
            start = time.perf_counter()
            cur.execute(sql, params)
            cur.fetchall()
            times.append((time.perf_counter() - start) * 1000)
    finally:
        conn.close()
    avg = sum(times) / len(times)
    print(f"⏱️  {label}: {avg:.2f} ms (avg of {runs} runs, warm cache)")
    return avg

# Test connection
try:
    conn = get_conn()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ Connection failed: {e}")

---

## 🔴 BAD: One Huge Table (No Partitioning)

Let's create an `events` table with **1 million rows** of time-series data. This simulates a logging/analytics table that grows every day.

Without partitioning, every query must scan or index the entire table.

In [ ]:
# Create a single large events table (no partitioning)
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS events CASCADE")
cur.execute("DROP TABLE IF EXISTS events_range CASCADE")
cur.execute("DROP TABLE IF EXISTS events_list CASCADE")
cur.execute("DROP TABLE IF EXISTS events_hash CASCADE")

cur.execute("""
    CREATE TABLE events (
        id SERIAL PRIMARY KEY,
        user_id INTEGER NOT NULL,
        event_type VARCHAR(50) NOT NULL,
        payload JSONB,
        region VARCHAR(20) NOT NULL,
        created_at TIMESTAMP NOT NULL
    )
""")

# Generate 1 million events spanning 12 months
cur.execute("""
    INSERT INTO events (user_id, event_type, payload, region, created_at)
    SELECT
        (floor(random() * 5000) + 1)::int,
        (ARRAY['page_view', 'click', 'purchase', 'signup', 'logout'])[floor(random() * 5 + 1)::int],
        jsonb_build_object('source', 'web', 'session', md5(random()::text)),
        (ARRAY['us-east', 'us-west', 'eu-west', 'eu-east', 'ap-south'])[floor(random() * 5 + 1)::int],
        NOW() - (random() * interval '365 days')
    FROM generate_series(1, 1000000)
""")

cur.execute("CREATE INDEX idx_events_created ON events(created_at)")
cur.execute("CREATE INDEX idx_events_type ON events(event_type)")
cur.execute("ANALYZE events")

cur.execute("SELECT COUNT(*) FROM events")
count = cur.fetchone()[0]
conn.close()

print(f"✅ Created events table: {count:,} rows")
print()

# Show table size
size_info, _ = run_sql("""
    SELECT
        pg_size_pretty(pg_total_relation_size('events')) AS total_size,
        pg_size_pretty(pg_relation_size('events')) AS table_size,
        pg_size_pretty(pg_indexes_size('events')) AS index_size
""")
print(f"📏 Table size: {size_info[0][1]}")
print(f"📏 Index size: {size_info[0][2]}")
print(f"📏 Total size: {size_info[0][0]}")

In [ ]:
# Query: Get events from the last 7 days

print("=" * 60)
print("🔴 BAD: Query on a single huge table (1M rows)")
print("=" * 60)
print()

query_7d = "SELECT COUNT(*) FROM events WHERE created_at > NOW() - INTERVAL '7 days'"
query_type = "SELECT COUNT(*) FROM events WHERE event_type = 'purchase' AND created_at > NOW() - INTERVAL '30 days'"

bad_t1 = timed_query(query_7d, label="Events in last 7 days")
bad_t2 = timed_query(query_type, label="Purchases in last 30 days")
print()

explain(query_7d)
print()
print("💡 Even with an index, the query has to scan all matching rows.")
print("   As the table grows to 100M+ rows, this gets MUCH slower.")

---

## 🟡 BETTER: Range Partitioning (by date)

**Range partitioning** splits the table by value ranges — most commonly by date. Each partition holds data for a specific time period (e.g., one month).

```
events_range (parent — no data)
├── events_2025_q1  (Jan-Mar 2025)
├── events_2025_q2  (Apr-Jun 2025)
├── events_2025_q3  (Jul-Sep 2025)
├── events_2025_q4  (Oct-Dec 2025)
├── events_2026_q1  (Jan-Mar 2026)
└── events_2026_q2  (Apr-Jun 2026)
```

When you query `WHERE created_at > '2026-03-25'`, PostgreSQL only scans the Q1/Q2 2026 partitions and **skips everything else**. This is **partition pruning**.

In [ ]:
# Create a range-partitioned version of the same table
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS events_range CASCADE")

cur.execute("""
    CREATE TABLE events_range (
        id SERIAL,
        user_id INTEGER NOT NULL,
        event_type VARCHAR(50) NOT NULL,
        payload JSONB,
        region VARCHAR(20) NOT NULL,
        created_at TIMESTAMP NOT NULL
    ) PARTITION BY RANGE (created_at)
""")

# Create quarterly partitions for 2025 and 2026
partitions = [
    ("events_range_2025q1", "'2025-01-01'", "'2025-04-01'"),
    ("events_range_2025q2", "'2025-04-01'", "'2025-07-01'"),
    ("events_range_2025q3", "'2025-07-01'", "'2025-10-01'"),
    ("events_range_2025q4", "'2025-10-01'", "'2026-01-01'"),
    ("events_range_2026q1", "'2026-01-01'", "'2026-04-01'"),
    ("events_range_2026q2", "'2026-04-01'", "'2026-07-01'"),
]

for name, start, end in partitions:
    cur.execute(f"""
        CREATE TABLE {name} PARTITION OF events_range
        FOR VALUES FROM ({start}) TO ({end})
    """)

# Copy data from the unpartitioned table
cur.execute("""
    INSERT INTO events_range (id, user_id, event_type, payload, region, created_at)
    SELECT id, user_id, event_type, payload, region, created_at
    FROM events
    WHERE created_at >= '2025-01-01' AND created_at < '2026-07-01'
""")

# Create indexes on each partition (done automatically on parent)
cur.execute("CREATE INDEX ON events_range(created_at)")
cur.execute("CREATE INDEX ON events_range(event_type, created_at)")
cur.execute("ANALYZE events_range")

cur.execute("SELECT COUNT(*) FROM events_range")
count = cur.fetchone()[0]
conn.close()

print(f"✅ Created range-partitioned table: {count:,} rows across {len(partitions)} partitions")
print()

# Show partition sizes
parts, cols = run_sql("""
    SELECT
        child.relname AS partition,
        pg_size_pretty(pg_relation_size(child.oid)) AS size,
        (SELECT COUNT(*) FROM events_range WHERE tableoid = child.oid) AS rows
    FROM pg_inherits
    JOIN pg_class child ON child.oid = pg_inherits.inhrelid
    JOIN pg_class parent ON parent.oid = pg_inherits.inhparent
    WHERE parent.relname = 'events_range'
    ORDER BY child.relname
""")

print("📏 Partition Sizes:")
print(tabulate(parts, headers=cols, tablefmt="simple_grid"))

In [ ]:
# Now query the partitioned table — watch for "Partition Pruning"!

print("=" * 60)
print("🟡 BETTER: Range partitioning — partition pruning in action!")
print("=" * 60)
print()

query_range = "SELECT COUNT(*) FROM events_range WHERE created_at > NOW() - INTERVAL '7 days'"

explain(query_range)
print()

better_t1 = timed_query(query_range, label="Events in last 7 days (range partitioned)")
print()
print(f"🚀 Speedup vs unpartitioned: {bad_t1 / better_t1:.1f}×")
print()
print("💡 Notice in the EXPLAIN output:")
print("   - Only 1-2 partitions are scanned (the recent ones)")
print("   - The other 4 partitions are completely skipped!")
print("   - This is 'partition pruning' — huge performance win")

---

## List Partitioning (by category)

**List partitioning** splits the table by specific values — like region or event type. This is great when queries always filter by a categorical column.

```
events_list (parent)
├── events_us_east   (region = 'us-east')
├── events_us_west   (region = 'us-west')
├── events_eu_west   (region = 'eu-west')
├── events_eu_east   (region = 'eu-east')
└── events_ap_south  (region = 'ap-south')
```

In [ ]:
# Create a list-partitioned table by region
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS events_list CASCADE")

cur.execute("""
    CREATE TABLE events_list (
        id SERIAL,
        user_id INTEGER NOT NULL,
        event_type VARCHAR(50) NOT NULL,
        payload JSONB,
        region VARCHAR(20) NOT NULL,
        created_at TIMESTAMP NOT NULL
    ) PARTITION BY LIST (region)
""")

regions = {
    "events_list_us_east": "us-east",
    "events_list_us_west": "us-west",
    "events_list_eu_west": "eu-west",
    "events_list_eu_east": "eu-east",
    "events_list_ap_south": "ap-south",
}

for name, region in regions.items():
    cur.execute(f"""
        CREATE TABLE {name} PARTITION OF events_list
        FOR VALUES IN ('{region}')
    """)

# Copy data
cur.execute("""
    INSERT INTO events_list (id, user_id, event_type, payload, region, created_at)
    SELECT id, user_id, event_type, payload, region, created_at
    FROM events
""")

cur.execute("CREATE INDEX ON events_list(created_at)")
cur.execute("ANALYZE events_list")
conn.close()

print("✅ Created list-partitioned table (by region)")
print()

# Query for a specific region
print("=" * 60)
print("📋 List Partitioning — query by region")
print("=" * 60)
print()

explain("SELECT COUNT(*) FROM events_list WHERE region = 'us-east'")
print()

list_t = timed_query(
    "SELECT COUNT(*) FROM events_list WHERE region = 'us-east'",
    label="Events in us-east (list partitioned)"
)

# Compare with unpartitioned
nop_t = timed_query(
    "SELECT COUNT(*) FROM events WHERE region = 'us-east'",
    label="Events in us-east (NOT partitioned)"
)

print()
print(f"🚀 Speedup: {nop_t / list_t:.1f}×")
print()
print("💡 List partitioning is great when you ALWAYS filter by the partition key.")
print("   Each region's data is completely isolated — fast scans, easy maintenance.")

---

## Hash Partitioning (even distribution)

**Hash partitioning** distributes rows evenly across partitions using a hash function. It doesn't help with range queries, but it's great for spreading data evenly when there's no natural partition key.

```
events_hash (parent)
├── events_hash_0  (hash(user_id) % 4 = 0)
├── events_hash_1  (hash(user_id) % 4 = 1)
├── events_hash_2  (hash(user_id) % 4 = 2)
└── events_hash_3  (hash(user_id) % 4 = 3)
```

In [ ]:
# Create a hash-partitioned table by user_id
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS events_hash CASCADE")

cur.execute("""
    CREATE TABLE events_hash (
        id SERIAL,
        user_id INTEGER NOT NULL,
        event_type VARCHAR(50) NOT NULL,
        payload JSONB,
        region VARCHAR(20) NOT NULL,
        created_at TIMESTAMP NOT NULL
    ) PARTITION BY HASH (user_id)
""")

# Create 4 hash partitions
for i in range(4):
    cur.execute(f"""
        CREATE TABLE events_hash_{i} PARTITION OF events_hash
        FOR VALUES WITH (MODULUS 4, REMAINDER {i})
    """)

# Copy data
cur.execute("""
    INSERT INTO events_hash (id, user_id, event_type, payload, region, created_at)
    SELECT id, user_id, event_type, payload, region, created_at
    FROM events
""")

cur.execute("CREATE INDEX ON events_hash(user_id)")
cur.execute("ANALYZE events_hash")
conn.close()

print("✅ Created hash-partitioned table (by user_id, 4 partitions)")
print()

# Check even distribution
parts, cols = run_sql("""
    SELECT
        child.relname AS partition,
        pg_size_pretty(pg_relation_size(child.oid)) AS size,
        (SELECT COUNT(*) FROM events_hash WHERE tableoid = child.oid) AS rows
    FROM pg_inherits
    JOIN pg_class child ON child.oid = pg_inherits.inhrelid
    JOIN pg_class parent ON parent.oid = pg_inherits.inhparent
    WHERE parent.relname = 'events_hash'
    ORDER BY child.relname
""")

print("📏 Hash Partition Distribution:")
print(tabulate(parts, headers=cols, tablefmt="simple_grid"))
print()

# Query for a specific user — only 1 partition scanned
print("=" * 60)
print("🔢 Hash Partitioning — query by user_id")
print("=" * 60)
print()

explain("SELECT COUNT(*) FROM events_hash WHERE user_id = 42")
print()

hash_t = timed_query(
    "SELECT COUNT(*) FROM events_hash WHERE user_id = 42",
    label="Events for user 42 (hash partitioned)"
)

nop_user_t = timed_query(
    "SELECT COUNT(*) FROM events WHERE user_id = 42",
    label="Events for user 42 (NOT partitioned)"
)

print()
print(f"🚀 Speedup: {nop_user_t / hash_t:.1f}×")
print()
print("💡 Hash partitioning gives even distribution across partitions.")
print("   Lookups by the hash key scan only 1 partition (1/4 of data).")

---

## 📊 Comparison: Which Partitioning Strategy to Use?

| Strategy | Best For | Partition Key | Pruning Works When |
|----------|---------|---------------|-------------------|
| **Range** | Time-series data, logs | Date/timestamp | `WHERE created_at > X` |
| **List** | Regional data, categories | Status, region, type | `WHERE region = 'us-east'` |
| **Hash** | Even distribution, user data | Any column | `WHERE user_id = X` |

In [ ]:
# Final comparison: all strategies side by side

print("=" * 60)
print("📊 FINAL COMPARISON: All Partitioning Strategies")
print("=" * 60)
print()

# Time-based query (range partitioning shines)
q_time = "SELECT COUNT(*) FROM {} WHERE created_at > NOW() - INTERVAL '7 days'"

t_none_time = timed_query(q_time.format("events"), label="🔴 No partitioning (time query)")
t_range_time = timed_query(
    "SELECT COUNT(*) FROM events_range WHERE created_at > NOW() - INTERVAL '7 days'",
    label="🟢 Range partition (time query)"
)

print()

# Region-based query (list partitioning shines)
q_region = "SELECT COUNT(*) FROM {} WHERE region = 'us-east'"

t_none_region = timed_query(q_region.format("events"), label="🔴 No partitioning (region query)")
t_list_region = timed_query(q_region.format("events_list"), label="🟢 List partition (region query)")

print()

# User-based query (hash partitioning shines)
q_user = "SELECT COUNT(*) FROM {} WHERE user_id = 42"

t_none_user = timed_query(q_user.format("events"), label="🔴 No partitioning (user query)")
t_hash_user = timed_query(q_user.format("events_hash"), label="🟢 Hash partition (user query)")

print()
print("=" * 60)
print("📊 SUMMARY")
print("=" * 60)
print()
results = [
    ["Time range query", f"{t_none_time:.2f} ms", f"{t_range_time:.2f} ms", f"{t_none_time/t_range_time:.1f}×", "Range"],
    ["Region query", f"{t_none_region:.2f} ms", f"{t_list_region:.2f} ms", f"{t_none_region/t_list_region:.1f}×", "List"],
    ["User lookup", f"{t_none_user:.2f} ms", f"{t_hash_user:.2f} ms", f"{t_none_user/t_hash_user:.1f}×", "Hash"],
]
print(tabulate(results,
               headers=["Query Type", "No Partition", "Partitioned", "Speedup", "Best Strategy"],
               tablefmt="simple_grid"))

## 🧹 Cleanup

In [ ]:
# Clean up all the tables created in this notebook
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
for table in ['events', 'events_range', 'events_list', 'events_hash']:
    cur.execute(f"DROP TABLE IF EXISTS {table} CASCADE")
conn.close()
print("🧹 All event tables removed")

## 📚 Summary

### Key Takeaways

1. **Partitioning splits big tables** into smaller pieces for faster queries
2. **Partition pruning** lets PostgreSQL skip irrelevant partitions entirely
3. **Range partitioning** is best for time-series data (logs, events, metrics)
4. **List partitioning** is best for categorical data (regions, statuses)
5. **Hash partitioning** is best for even distribution (user IDs)
6. **Choose your partition key based on your query patterns** — partition on the column you filter by most

### When to Partition

| Table Size | Recommendation |
|-----------|---------------|
| < 1M rows | Don't partition — indexes are enough |
| 1M - 100M rows | Consider partitioning if queries are slow |
| > 100M rows | Almost always partition |

### Partitioning in System Design Interviews

- Mention partitioning when you have **time-series data** or **very large tables**
- Explain which strategy you'd use and **why** (based on query patterns)
- Remember: partitioning helps reads but adds complexity to schema management

### 🎉 Congratulations!

You've completed the PostgreSQL Deep Dive lab! You now understand:
1. **Indexing** — B-tree, composite, partial, covering indexes
2. **Query optimization** — N+1, JOINs, window functions
3. **Replication** — streaming replication, read scaling, failover
4. **Partitioning** — range, list, hash strategies